# GX_04 — Predicting Who Will Leave (Modelling)

**The question this notebook answers:** *Which customers are most likely to leave the bank next, and how well can we predict that before it happens?*

### What this notebook is about
GX_03 described the customers who left. Here we build models that learn from them and give every customer a **churn risk score** between 0 and 1, so the bank can talk to the riskiest customers first.

In this notebook we:
1. **Check** the fixed train / validation / test split that GX_02 built, and reuse the shared preparation steps.
2. **Compare** three baseline models with default settings: logistic regression, Random Forest and Gradient-Boosted Trees.
3. **Tune** each model with a small **grid search** — every combination of the chosen hyperparameters, scored by 3-fold cross-validation inside the training split.
4. **Stack** the two best models into one ensemble, and check honestly whether that helps.
5. **Evaluate** the chosen approach on the test set — once, at the very end.

### What this notebook uses and produces
| | |
|---|---|
| **Reads** | `gold_churn_features` (10,000 customers, with the fixed `split` column) |
| **Writes** | No new tables. The output is the chosen model, its scores and the conclusions. |

### Three rules for reading the results
- **The test set is opened once, at the very end.** Every comparison and every decision before that uses only the training and validation customers. This is what makes the final test number honest.
- **Training scores are not model selection.** A model can memorise the training rows (the tree models especially) and still fail on new customers. Selection always uses the **validation** score.
- **Every model is judged on the same primary metric: AUC-PR**, the metric GX_03 chose for this problem (Step 2 below explains why, in plain words).

## Step 0 — Preparing the notebook

**What we do:** load the shared helper code from `src/`, one colour palette for every chart, and the model classes we need.

**Why:**
- The preparation steps (text → numbers, scaling) are exactly the ones defined in `src/ml.py` in Phase 2. We import and reuse them instead of writing new ones, so the model sees the same 16 inputs GX_02 planned — nothing is re-invented, and nothing can drift between phases.
- `SEED = 42` from `src/config.py` is used for every random operation (tree building, cross-validation folds), so running the notebook again gives the same results.
- The colours match the other notebooks, so the slides all look alike.
- **One environment setting:** Spark ML's `CrossValidator` (Steps 4–6) caches intermediate DataFrames. On serverless compute that cache must live in a **Unity Catalog volume**, so the setup cell creates an empty scratch volume (`ml_tmp`) and points Spark ML at it via the `SPARKML_TEMP_DFS_PATH` environment variable. The volume holds only temporary Spark files, never project data.

In [0]:
# Make the shared helpers in src/ importable (this notebook lives in notebooks/).
import os
import sys

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier, LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import Window
from pyspark.sql import functions as F

from src.config import EXPECTED_ROWS, GOLD_CHURN, SEED
from src.ml import (
    CATEGORICAL_FEATURES, KEY_COL, LABEL_COL, NUMERIC_FEATURES, SPLIT_COL,
    build_preprocessing_stages,
)

# One colour palette for every chart (same as the other notebooks)
COLOR_MAIN = "#27ae60"    # general bars / validation scores
COLOR_DARK = "#0b5345"    # highlight / training scores
COLOR_GREY = "#7f8c8d"    # reference line

print("Spark version:", spark.version)

# CrossValidator (Steps 4-6) caches intermediate DataFrames. On serverless compute
# Spark ML needs a Unity Catalog volume for that cache, set via this environment
# variable. The volume holds only temporary Spark files, never project data.
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.churn.ml_tmp")
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/churn/ml_tmp"
print("Spark ML temp path:", os.environ["SPARKML_TEMP_DFS_PATH"])

## Step 1 — The data and the fixed split

**What we do:** read `gold_churn_features`, split it into the three parts GX_02 assigned (`train` / `val` / `test`), and check the split before any model is fitted.

**Why check again, if GX_02 already did:** the model notebook is exactly where a broken split would do the most damage. If the parts overlapped, the same customer could be used both for learning and for judging, and every score would look better than it really is. We verify three things:
- every customer appears in **exactly one** part (no overlap),
- the three parts together contain **all 10,000** customers (nothing lost, nothing duplicated),
- the **churn rate is about the same** in all three parts (the stratification GX_02 built worked).

**Why reuse the split instead of making a new one:** GX_02 fixed it once, deterministically (a hash of the customer id, not a random draw), and stored it in the table. Every notebook — and every model in this one — now uses the same customers in the same role, so scores are comparable and the test customers stay unseen by anything that learns.

**Why `cache()`:** the three parts are used many times below (training, cross-validation folds, scoring). `cache()` keeps them in memory after the first use, so Spark does not rebuild them each time.

In [0]:
gold = spark.table(GOLD_CHURN).cache()
gold.createOrReplaceTempView("gold")

train = gold.filter(F.col(SPLIT_COL) == "train")
val = gold.filter(F.col(SPLIT_COL) == "val")
test = gold.filter(F.col(SPLIT_COL) == "test")

display(spark.sql(f'''
    SELECT {SPLIT_COL} AS split,
           COUNT(*)                         AS customers,
           SUM({LABEL_COL})                 AS churned,
           ROUND(AVG({LABEL_COL}) * 100, 2) AS churn_rate_pct
    FROM gold
    GROUP BY {SPLIT_COL}
    ORDER BY CASE {SPLIT_COL} WHEN 'train' THEN 1 WHEN 'val' THEN 2 ELSE 3 END
'''))

# Overlap check: the same customer must never appear in two splits.
total_rows = gold.count()
distinct_customers = gold.select(KEY_COL).distinct().count()
customers_in_two_splits = (
    gold.groupBy(KEY_COL)
    .agg(F.count_distinct(SPLIT_COL).alias("n_splits"))
    .filter(F.col("n_splits") > 1)
    .count()
)

print(f"Rows: {total_rows:,} | Distinct customers: {distinct_customers:,}")
print(f"Customers appearing in more than one split: {customers_in_two_splits}")
assert (
    total_rows == distinct_customers == EXPECTED_ROWS["churn"] and customers_in_two_splits == 0
), "Split broken - stop before modelling."

**Result:** the split is exactly the one GX_02 built — same sizes, same churn rates, no overlap.

| Split | Customers | Leavers | Churn rate |
|---|---|---|---|
| train | 7,000 | 1,426 | 20.4% |
| validation | 1,499 | 305 | 20.4% |
| test | 1,501 | 306 | 20.4% |

10,000 rows, 10,000 distinct customers, and **0 customers in more than one split** — the three parts are genuinely separate. The churn rate is ~20.4% everywhere (the split is stratified), so the class imbalance is the same for training and for measuring, and a random model's AUC-PR would be ≈ 0.204 in all three parts. Everything below can now be compared like for like.

## Step 2 — How we judge the models: one metric for everyone

**What we do:** fix **AUC-PR** (area under the precision–recall curve) as the primary metric for every model in this notebook — baselines, grid searches and the final choice.

**Why AUC-PR for this problem:**
- Only about **20.4%** of customers leave. A model that says "stays" for *everyone* is already 79.6% accurate, so **accuracy would flatter any model here** — it says nothing about finding the one customer in five who actually leaves. This is why GX_03 already rejected it.
- **Precision–recall looks only at the churn class.** Precision: of the customers we flag as risky, how many really leave. Recall: of the leavers, how many we catch. AUC-PR summarises every possible trade-off between the two into one number, without letting the 7,963 stayers dominate the score.
- **A random model scores about 0.20** on AUC-PR (roughly the churn rate); a perfect model scores 1.0. Every number in this notebook can be read against that scale, and we draw it as a dashed reference line in the charts.
- The business objective is a **ranked list**: the retention team cannot call everyone, so it wants the true leavers as close to the top as possible. AUC-PR measures exactly the quality of that ranking — and it does not depend on where we later set the contact threshold.

**Complementary metrics.** A ranking score alone cannot tell the bank how many retention offers would be wasted or missed, so for every *selected* model we also report, at the default threshold 0.5: **precision, recall and F1 for the churn class**, and the **confusion matrix**. The F1 is computed **by hand from the confusion matrix and is for the churn class only** — Spark's built-in F1 evaluator averages both classes, which would quietly hide exactly the group the bank cares about.

**Threshold.** Spark classifies a customer as "churn" when the probability is 0.5 or more (the default). We keep that threshold everywhere and **freeze it before the final test evaluation**; any threshold choice is a modelling decision, and we make ours before ever looking at the test set.

In [0]:
auc_pr_eval = BinaryClassificationEvaluator(
    labelCol=LABEL_COL, rawPredictionCol="probability", metricName="areaUnderPR"
)
auc_roc_eval = BinaryClassificationEvaluator(
    labelCol=LABEL_COL, rawPredictionCol="probability", metricName="areaUnderROC"
)

# A random model scores an AUC-PR roughly equal to the churn rate (GX_03: 20.4%).
NO_SKILL_AUC_PR = round(train.agg(F.avg(LABEL_COL)).first()[0], 4)
print(f"Reference line for the charts (random model): {NO_SKILL_AUC_PR}")

results = []   # one row per model and stage ("baseline" / "tuned"); filled throughout


def make_pipeline(classifier):
    """GX_02's preparation steps + one classifier, as one Spark ML Pipeline.

    The preparation stages are UNFITTED: the pipeline learns them from whatever rows
    .fit() is called on. Fitted on `train`, the indexer and scaler see only the 7,000
    training rows; inside cross-validation each fold fits them again on that fold's
    training rows only. Validation and test rows never teach the preprocessing anything.
    """
    return Pipeline(
        stages=build_preprocessing_stages(CATEGORICAL_FEATURES, NUMERIC_FEATURES) + [classifier]
    )


def churn_metrics(preds):
    """All metrics of a prediction DataFrame (needs label, probability, prediction).

    Precision / recall / F1 are for the CHURN class only, computed by hand from the
    confusion matrix at Spark's default threshold (0.5).
    """
    df = preds.withColumn("p_churn", vector_to_array("probability")[1].cast("double"))
    cm = df.agg(
        F.sum(((F.col(LABEL_COL) == 1) & (F.col("prediction") == 1)).cast("int")).alias("tp"),
        F.sum(((F.col(LABEL_COL) == 0) & (F.col("prediction") == 1)).cast("int")).alias("fp"),
        F.sum(((F.col(LABEL_COL) == 1) & (F.col("prediction") == 0)).cast("int")).alias("fn"),
        F.sum(((F.col(LABEL_COL) == 0) & (F.col("prediction") == 0)).cast("int")).alias("tn"),
    ).first()
    tp, fp, fn, tn = (int(cm[c]) for c in ("tp", "fp", "fn", "tn"))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {
        "auc_pr": round(auc_pr_eval.evaluate(df), 4),
        "auc_roc": round(auc_roc_eval.evaluate(df), 4),
        "precision_churn": round(precision, 4),
        "recall_churn": round(recall, 4),
        "f1_churn": round(2 * precision * recall / (precision + recall), 4) if precision + recall else 0.0,
        "confusion": {"TP": tp, "FP": fp, "FN": fn, "TN": tn},
    }


def score_row(model_name, train_preds, val_preds, stage):
    """One comparison row: train and val AUC-PR, the gap, and validation extras."""
    m_val = churn_metrics(val_preds)
    row = {
        "model": model_name,
        "stage": stage,
        "train_auc_pr": round(auc_pr_eval.evaluate(train_preds), 4),
        "val_auc_pr": m_val["auc_pr"],
        "train_val_gap": None,   # filled on the next line
        "val_auc_roc": m_val["auc_roc"],
        "val_precision_churn": m_val["precision_churn"],
        "val_recall_churn": m_val["recall_churn"],
        "val_f1_churn": m_val["f1_churn"],
    }
    row["train_val_gap"] = round(row["train_auc_pr"] - row["val_auc_pr"], 4)
    results.append(row)
    return row


def compare_table(name):
    """Baseline and tuned rows of one model, for the before/after sections."""
    return pd.DataFrame([r for r in results if r["model"] == name])


def show_confusion(preds, title):
    """Print the churn-class metrics and show the confusion matrix (rows = prediction)."""
    m = churn_metrics(preds)
    print(title, "-", {k: v for k, v in m.items() if k != "confusion"})
    cm = m["confusion"]
    display(spark.createDataFrame(
        [(0, cm["TN"], cm["FP"]), (1, cm["FN"], cm["TP"])],
        ["predicted", "actually_stayed", "actually_churned"],
    ))


def plot_train_scores(ax, names, scores, title):
    """One bar per model, the score written above each bar, the no-skill line dashed."""
    bars = ax.bar(names, scores, color=COLOR_MAIN)
    for bar, s in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{s:.3f}",
                ha="center", va="bottom", fontsize=10)
    ax.axhline(NO_SKILL_AUC_PR, color=COLOR_GREY, linestyle="--", linewidth=1,
               label=f"Random model ({NO_SKILL_AUC_PR:.3f})")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("AUC-PR")
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=9)


def plot_train_val(ax, groups, train_scores, val_scores, title):
    """Two bars per group (training dark, validation green), scores on top."""
    x = range(len(groups))
    width = 0.4
    for offset, values, color in [(-width / 2, train_scores, COLOR_DARK),
                                  (width / 2, val_scores, COLOR_MAIN)]:
        bars = ax.bar([i + offset for i in x], values, width, color=color)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + width / 2, bar.get_height() + 0.01, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=9)
    ax.axhline(NO_SKILL_AUC_PR, color=COLOR_GREY, linestyle="--", linewidth=1)
    ax.set_xticks(list(x))
    ax.set_xticklabels(groups)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("AUC-PR")
    ax.set_title(title)
    ax.legend(handles=[
        Patch(color=COLOR_DARK, label="training"),
        Patch(color=COLOR_MAIN, label="validation"),
        Line2D([0], [0], color=COLOR_GREY, linestyle="--",
               label=f"Random model ({NO_SKILL_AUC_PR:.3f})"),
    ], loc="lower right", fontsize=9)

## Step 3 — Baseline: three models with default settings

**What we do:** train one version of each model with Spark's default settings and our fixed preparation steps, all fitted on the 7,000 training rows.

| Model | What it is, in one sentence |
|---|---|
| **Logistic regression** | One weighted sum of the inputs, pushed through a curve to give a churn probability; draws a single straight decision boundary. |
| **Random Forest** | Many decision trees, each fitted on a random sample of rows and inputs; their votes are averaged. |
| **Gradient-Boosted Trees (GBT)** | Decision trees fitted one after another, each new tree correcting the mistakes of the trees so far. |

**Why start with defaults:** a baseline with default settings is a fair, reproducible starting point. It shows what each model family can do "out of the box", and how much the small grid searches in Steps 4–6 actually add. These are **baseline results, before any hyperparameter tuning**.

**What "fitting on the training rows" means here:** the *whole pipeline* — the text→number and scaling steps from GX_02 *and* the model — is fitted on `train` only. Validation rows are only ever passed through the finished pipeline to be scored.

**Note on the model inputs:** the features are exactly the 16 columns GX_02 prepared: `Geography` and `Gender` (one-hot encoded) plus the 11 number columns. The target `Exited`, the key `CustomerId`, the `split` label and the text group columns (`AgeGroup`, …) are **not** inputs — the key would let the model memorise customers, and the group columns just repeat information already in the numbers.

In [0]:
baseline_clfs = [
    ("Logistic Regression", LogisticRegression(labelCol=LABEL_COL)),
    ("Random Forest", RandomForestClassifier(labelCol=LABEL_COL, seed=SEED)),
    ("Gradient-Boosted Trees", GBTClassifier(labelCol=LABEL_COL, seed=SEED)),
]

baseline_models = {}
for name, clf in baseline_clfs:
    baseline_models[name] = make_pipeline(clf).fit(train)
    score_row(name, baseline_models[name].transform(train), baseline_models[name].transform(val), "baseline")
    print(f"{name}: trained and scored.")

baseline_table = pd.DataFrame([r for r in results if r["stage"] == "baseline"])
display(spark.createDataFrame(
    baseline_table[["model", "train_auc_pr", "val_auc_pr", "train_val_gap"]]
))

In [0]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_train_scores(
    ax,
    list(baseline_table["model"]),
    list(baseline_table["train_auc_pr"]),
    "Baseline models (default settings): training AUC-PR — before tuning",
)
plt.tight_layout()
plt.show()

**Result:** all three models already beat the random reference line (0.204) by a wide margin with default settings.

| Model | Training AUC-PR | Validation AUC-PR | Gap |
|---|---|---|---|
| Logistic regression | 0.631 | 0.604 | 0.027 |
| Random Forest | 0.686 | 0.638 | 0.048 |
| Gradient-Boosted Trees | 0.771 | 0.656 | 0.115 |

- **Every model generalises:** validation stays far above the random line — the 16 inputs carry real churn signal.
- **GBT is the strongest on validation (0.656)**, but already shows the largest gap (0.115): the early memorising risk that Step 6 addresses.
- **Logistic regression is clearly last (0.604)** — exactly what GX_03 predicted for a model that can only draw one straight boundary (the U-shaped product pattern). Its tiny gap (0.027) fits: a simple model has little room to memorise.

**How to read this, before going on:**
- The chart shows **training** scores only: how well each model reproduces the rows it was fitted on. **That number must not be used to pick a model.** A flexible model can memorise the training rows and still fail on customers it has not seen — the table's validation column is the honest one.
- The **train–validation gap** is our first overfitting signal: a large positive gap means the model does much better on rows it learned from than on new rows. A small gap is *not* automatically good news either — a weak model can have almost no gap. What matters is the validation performance, with the gap as context.
- These are **baseline results, before hyperparameter tuning** — the point is a fair starting point, not a final choice.

## Step 4 — Model 1: Logistic regression, tuned

**How logistic regression works (briefly).** The model computes one weighted sum of the 16 inputs — age × w₁ + balance × w₂ + … — adds a constant, and pushes the result through the logistic curve, which turns any number into a probability between 0 and 1. Fitting means choosing the weights that give the observed churners high probabilities. It is the most interpretable of our three models (each weight says in which direction and how strongly an input pushes the risk), which is why banks like it as a reference. Its limit, already seen in GX_03: it can only draw **one straight boundary**, so patterns like “2 products is the safe spot” (risk goes down *and then up*) are invisible to it.

**Baseline result (Step 3):** training AUC-PR 0.631, validation 0.604 (gap 0.027) — the weakest of the three, but very stable, as expected for a model with only one straight boundary.

**What we tune, and why — the grid search:**
- `regParam` (0, 0.01, 0.1): how strongly the weights are shrunk towards zero. More shrinkage = a simpler, more stable model; 0 = no shrinkage at all.
- `elasticNetParam` (0, 0.5): the mix between ridge (shrinks every weight a little) and lasso (can shrink a weight to exactly zero, dropping the input).

Six combinations — the full grid — judged with **3-fold cross-validation inside the training split** (18 quick fits): the 7,000 training rows are cut into three parts; each candidate is trained on two parts and scored on the third, three times over, and the three scores are averaged. The preparation steps sit **inside the pipeline**, so within every fold the indexer and scaler are fitted on that fold’s training rows only — nothing leaks from the fold being scored. The winner is then refitted on all 7,000 training rows and scored once on the validation split. All 18 candidates are judged on the same AUC-PR, with a fixed seed.

In [0]:
lr = LogisticRegression(labelCol=LABEL_COL)
lr_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.0, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5])
    .build()
)

lr_cv = CrossValidator(
    estimator=make_pipeline(lr),   # the SAME lr instance the grid refers to
    estimatorParamMaps=lr_grid,
    evaluator=auc_pr_eval,         # the same primary metric as everywhere else
    numFolds=3,
    seed=SEED,
)
lr_cv_model = lr_cv.fit(train)

# Grid-search result: every candidate's average AUC-PR across the three folds, best first.
lr_cv_table = pd.DataFrame(
    [
        {
            "regParam": params[lr.regParam],
            "elasticNetParam": params[lr.elasticNetParam],
            "cv_auc_pr": round(score, 4),
        }
        for params, score in zip(lr_grid, lr_cv_model.avgMetrics)
    ]
).sort_values("cv_auc_pr", ascending=False)
display(spark.createDataFrame(lr_cv_table))

lr_best = lr_cv_model.bestModel.stages[-1]
LR_BEST = {"regParam": lr_best.getRegParam(), "elasticNetParam": lr_best.getElasticNetParam()}
print(f"Selected parameters: regParam={LR_BEST['regParam']}, elasticNetParam={LR_BEST['elasticNetParam']}")

In [0]:
# Refit the selected settings on ALL 7,000 training rows, then score train and validation.
lr_tuned = make_pipeline(LogisticRegression(labelCol=LABEL_COL, **LR_BEST)).fit(train)
score_row("Logistic Regression", lr_tuned.transform(train), lr_tuned.transform(val), "tuned")

lr_compare = compare_table("Logistic Regression")
display(spark.createDataFrame(
    lr_compare[["stage", "train_auc_pr", "val_auc_pr", "train_val_gap"]]
))

fig, ax = plt.subplots(figsize=(8, 5))
plot_train_val(
    ax, ["Baseline", "Tuned"],
    list(lr_compare["train_auc_pr"]), list(lr_compare["val_auc_pr"]),
    "Logistic regression: AUC-PR before vs after tuning",
)
plt.tight_layout()
plt.show()

In [0]:
show_confusion(lr_tuned.transform(val), "Logistic regression (tuned) on validation")

**Result:** cross-validation found that **no shrinkage at all** is best: `regParam = 0`, `elasticNetParam = 0` (CV AUC-PR 0.624). Every shrinkage setting made the model worse — `regParam = 0.01` dropped the CV score to 0.548, and 0.1 to 0.496: the weights need their full size to describe the patterns. Because the winning settings are exactly Spark's defaults, the tuned model is identical to the baseline: **validation AUC-PR 0.604, gap 0.027**.

The honest verdict of Step 4: the search confirmed that logistic regression is already well configured — but it cannot reach the tree models (0.638–0.656 on validation). Its role in this project is the interpretable reference model, and it plays that role well. At threshold 0.5 on validation it catches 119 of the 305 leavers (recall 39%) at 68% precision (F1 0.497).

## Step 5 — Model 2: Random Forest, tuned

**How a random forest works (briefly).** A single decision tree asks a chain of questions (“age > 45?”, “products ≤ 1?”) until it can assign a risk. Trees are easy to read and naturally capture the U-shaped product effect, but one deep tree grown alone memorises noise. A **random forest** grows many trees, each on a random sample of the training rows and allowed to look only at a random subset of inputs at each question, then averages their answers. The averaging makes memorising much harder, and the randomness makes the trees different enough for averaging to help. Scaling is unnecessary for trees (a question compares one input to one threshold), but the shared pipeline still scales the inputs — that does no harm and keeps one pipeline for all three models.

**Baseline result (Step 3):** training AUC-PR 0.686, validation 0.638 (gap 0.048) — clearly ahead of logistic regression, as GX_03's hypothesis H2 predicted for the tree models.

**What we tune, and why — the grid search:**
- `numTrees` (20, 60): more trees = more averaging = a more stable forest, but slower. Spark's default is 20.
- `maxDepth` (5, 10): how many questions a tree may ask. Deeper = more interaction patterns (GX_03 found age × activity effects), but also a higher risk of memorising. Default 5.

Four combinations × 3 folds = 12 fits, the same grid-search rules as Step 4: preparation fitted inside each fold, AUC-PR as the judge, seed fixed, winner refitted on the full training split and scored once on validation.

In [0]:
rf = RandomForestClassifier(labelCol=LABEL_COL, seed=SEED)
rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 60])
    .addGrid(rf.maxDepth, [5, 10])
    .build()
)

rf_cv = CrossValidator(
    estimator=make_pipeline(rf),
    estimatorParamMaps=rf_grid,
    evaluator=auc_pr_eval,
    numFolds=3,
    seed=SEED,
)
rf_cv_model = rf_cv.fit(train)

# Grid-search result: every candidate's average AUC-PR across the three folds, best first.
rf_cv_table = pd.DataFrame(
    [
        {"numTrees": params[rf.numTrees], "maxDepth": params[rf.maxDepth],
         "cv_auc_pr": round(score, 4)}
        for params, score in zip(rf_grid, rf_cv_model.avgMetrics)
    ]
).sort_values("cv_auc_pr", ascending=False)
display(spark.createDataFrame(rf_cv_table))

rf_best = rf_cv_model.bestModel.stages[-1]
# numTrees is read with getOrDefault: on a fitted model it is a plain value, and this
# form works the same on every Spark version.
RF_BEST = {"numTrees": rf_best.getOrDefault("numTrees"), "maxDepth": rf_best.getMaxDepth()}
print(f"Selected parameters: numTrees={RF_BEST['numTrees']}, maxDepth={RF_BEST['maxDepth']}")

In [0]:
rf_tuned = make_pipeline(
    RandomForestClassifier(labelCol=LABEL_COL, seed=SEED, **RF_BEST)
).fit(train)
score_row("Random Forest", rf_tuned.transform(train), rf_tuned.transform(val), "tuned")

rf_compare = compare_table("Random Forest")
display(spark.createDataFrame(
    rf_compare[["stage", "train_auc_pr", "val_auc_pr", "train_val_gap"]]
))

fig, ax = plt.subplots(figsize=(8, 5))
plot_train_val(
    ax, ["Baseline", "Tuned"],
    list(rf_compare["train_auc_pr"]), list(rf_compare["val_auc_pr"]),
    "Random Forest: AUC-PR before vs after tuning",
)
plt.tight_layout()
plt.show()

show_confusion(rf_tuned.transform(val), "Random Forest (tuned) on validation")

**Result:** cross-validation chose the **biggest forest in the grid: 60 trees, depth 10** (CV AUC-PR 0.688, ahead of 0.674 for depth 5). Refitted on the full training split:
- Validation AUC-PR improved from **0.638 to 0.651** (+0.013).
- But the train–validation gap grew from **0.048 to 0.234**: a training score of 0.885 means the deep forest reproduces the rows it learned almost perfectly. That is the memorising the tuning rules warned about — the honest CV score (0.688, from rows the trees never saw) is the one that matches reality.
- At threshold 0.5 on validation: precision 76% (up from logistic regression's 68%), recall 42%, F1 0.540 — it catches 128 of the 305 leavers.

Verdict: tuning helped on validation, and the forest is a solid second — but of the three models, the tuned forest is the clearest overfitter. Its complexity (60 trees × depth 10) is also the hardest to explain.

## Step 6 — Model 3: Gradient-Boosted Trees, tuned

**How GBT works (briefly).** Boosting also combines trees, but in a different order: trees are fitted **one after another**, and each new tree concentrates on the rows the trees so far get wrong. The final score is the sum of all trees, each shrunk by a small learning rate (`stepSize`, default 0.1). Because every tree looks at the *remaining* mistakes, GBT often finds finer patterns than a forest — and, given too many trees or too much depth, it is also the quickest of the three to start memorising the training rows. GX_03 expects the tree models to beat logistic regression clearly (Hypothesis H2); this is where we check it.

**Baseline result (Step 3):** training AUC-PR 0.771, validation 0.656 (gap 0.115) — the best validation score of the three baselines, but with the largest gap.

**What we tune, and why — the grid search:**
- `maxIter` (20, 50): how many trees are added. Default 20.
- `maxDepth` (3, 5): deliberately shallower than the forest — boosting works best as many small corrections rather than a few big ones, and shallow trees are less prone to memorising.

Four combinations × 3 folds = 12 fits, the same grid-search rules as Steps 4 and 5.

In [0]:
gbt = GBTClassifier(labelCol=LABEL_COL, seed=SEED)
gbt_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxIter, [20, 50])
    .addGrid(gbt.maxDepth, [3, 5])
    .build()
)

gbt_cv = CrossValidator(
    estimator=make_pipeline(gbt),
    estimatorParamMaps=gbt_grid,
    evaluator=auc_pr_eval,
    numFolds=3,
    seed=SEED,
)
gbt_cv_model = gbt_cv.fit(train)

# Grid-search result: every candidate's average AUC-PR across the three folds, best first.
gbt_cv_table = pd.DataFrame(
    [
        {"maxIter": params[gbt.maxIter], "maxDepth": params[gbt.maxDepth],
         "cv_auc_pr": round(score, 4)}
        for params, score in zip(gbt_grid, gbt_cv_model.avgMetrics)
    ]
).sort_values("cv_auc_pr", ascending=False)
display(spark.createDataFrame(gbt_cv_table))

gbt_best = gbt_cv_model.bestModel.stages[-1]
GBT_BEST = {"maxIter": gbt_best.getMaxIter(), "maxDepth": gbt_best.getMaxDepth()}
print(f"Selected parameters: maxIter={GBT_BEST['maxIter']}, maxDepth={GBT_BEST['maxDepth']}")

In [0]:
gbt_tuned = make_pipeline(
    GBTClassifier(labelCol=LABEL_COL, seed=SEED, **GBT_BEST)
).fit(train)
score_row("Gradient-Boosted Trees", gbt_tuned.transform(train), gbt_tuned.transform(val), "tuned")

gbt_compare = compare_table("Gradient-Boosted Trees")
display(spark.createDataFrame(
    gbt_compare[["stage", "train_auc_pr", "val_auc_pr", "train_val_gap"]]
))

fig, ax = plt.subplots(figsize=(8, 5))
plot_train_val(
    ax, ["Baseline", "Tuned"],
    list(gbt_compare["train_auc_pr"]), list(gbt_compare["val_auc_pr"]),
    "Gradient-Boosted Trees: AUC-PR before vs after tuning",
)
plt.tight_layout()
plt.show()

show_confusion(gbt_tuned.transform(val), "Gradient-Boosted Trees (tuned) on validation")

**Result:** cross-validation chose **50 shallow trees (maxDepth 3)** — CV AUC-PR 0.703. Note what happened: the baseline default is depth 5, and CV preferred **shallower** trees. Refitted on the full training split:
- Validation AUC-PR **0.654**, essentially unchanged from the baseline 0.656 (−0.002, within noise).
- The train–validation gap **shrank from 0.115 to 0.079** (training 0.733): shallower trees memorise less. This is the other face of tuning — it did not buy performance, it bought **stability**.
- At threshold 0.5 on validation it has the best complementary numbers of the three models: precision 78%, recall 44%, F1 0.566.

Verdict: GBT stays the strongest single model, and after tuning it is also the most trustworthy of the tree models.

## Step 7 — The tuned models side by side

**What we do:** put the three tuned models next to each other — training and validation AUC-PR, the gap between them, and the validation precision / recall / F1 for the churn class — and choose the **two** models that Step 8 will combine into one ensemble.

**How we choose:**
1. The first criterion is the **validation AUC-PR**. Training scores are never used for selection — a model can be very good at the rows it memorised and still be the wrong choice.
2. When two models are close on validation, the **train–validation gap** (how much the model seems to have memorised) and the **complexity** (how many trees, how deep) break the tie: with similar, the simpler, less overfitting model is the safer bet on new customers.
3. We select two because Step 8 combines the two best into an ensemble — which works best when the two models make *different kinds of mistakes*, something the three model families here naturally do.

In [0]:
tuned_table = pd.DataFrame([r for r in results if r["stage"] == "tuned"])
display(spark.createDataFrame(tuned_table[[
    "model", "train_auc_pr", "val_auc_pr", "train_val_gap",
    "val_precision_churn", "val_recall_churn", "val_f1_churn",
]]))

fig, ax = plt.subplots(figsize=(9, 5))
plot_train_val(
    ax, list(tuned_table["model"]),
    list(tuned_table["train_auc_pr"]), list(tuned_table["val_auc_pr"]),
    "The three tuned models: training vs validation AUC-PR",
)
plt.tight_layout()
plt.show()

**Result — the two selected models: Gradient-Boosted Trees and Random Forest.**

| Tuned model | Validation AUC-PR | Gap | Precision | Recall | F1 (churn) |
|---|---|---|---|---|---|
| Gradient-Boosted Trees | **0.654** | 0.079 | 0.785 | 0.443 | 0.566 |
| Random Forest | 0.651 | 0.234 | 0.757 | 0.420 | 0.540 |
| Logistic regression | 0.604 | 0.027 | 0.684 | 0.390 | 0.497 |

- **GBT first:** the best validation AUC-PR and the smallest gap among the tree models; cross-validation also chose shallow, stable settings for it.
- **Random Forest second:** its validation score is only 0.003 behind, and its mistakes are different enough (many random trees averaged vs a sequence of corrections) to be worth combining. The large gap (0.234) is a caution, not a disqualification — the ensemble is judged on **its own** validation score in the next step.
- **Logistic regression is out of the ensemble:** 0.05 behind on validation. Its tiny gap is not a reason to keep it — a stable model can still be a weak one.

Both models' churn probabilities go into Step 8.

## Step 8 — Stacking the two best models

**What we do:** combine the two models selected in Step 7 — **Gradient-Boosted Trees** and **Random Forest** — into one ensemble, with a simple **logistic regression** as the *meta-model* that decides how to weigh their two churn probabilities.

**Why stacking, and why it must be built carefully.** Spark MLlib has no ready-made stacking estimator, so we build the standard *out-of-fold* version by hand. The idea: each base model gives a churn probability; the meta-model learns a good combination of the two (for example “mostly trust the GBT, but raise the risk when the forest also flags the customer”). The danger is **leakage**: if the meta-model is trained on predictions of base models *on the rows those models were fitted on*, it just learns “the base models are nearly always right” — an overfit base model reproduces its own training rows almost perfectly. The ensemble would look brilliant on those rows and collapse on new customers. Voting or averaging the probabilities would be a simpler ensemble, but it would not be stacking — it would ignore that the two models deserve different weights, and it could not learn them.

**How we prevent leakage (out-of-fold predictions):**
1. The 7,000 training rows are cut into **3 folds**, each with the same churn rate (the same deterministic trick as the GX_02 split).
2. For each fold, both base models are fitted on the **other two folds** — the preparation steps inside the pipeline are also fitted on those rows only — and predict the held-out fold. Every training row therefore gets two churn probabilities from models that **never saw that row**.
3. The meta-model (logistic regression) is trained on those out-of-fold probabilities and the true labels. It never sees a validation or test label.
4. To score *new* data (validation, later test), both base models are **refitted on the full training split**; their two probabilities become the meta-model's inputs, and the meta-model combines them.

**One detail — joining by `CustomerId`:** the two models' probabilities are matched by customer id. The id is only the *key* of the join; it is **never an input feature** — the meta-model's only inputs are the two probabilities.

**Honest expectation:** stacking only helps if the two models make *different* mistakes. We do not assume it will win — the comparison below shows whether it does, and if it does not, we report that plainly.

In [0]:
from pyspark.ml.feature import VectorAssembler

N_STACK_FOLDS = 3


def add_stack_fold(df, n_folds=N_STACK_FOLDS, seed=SEED):
    """Assign every training row to one of n folds (0..n-1), stratified by Exited.

    The same trick as src/ml.py's add_split_column: order rows by a hash of
    (CustomerId, a different seed) inside each label group, then cut the order into
    equal parts. Deterministic, and every fold keeps the same churn rate.
    """
    order = F.xxhash64(F.col(KEY_COL).cast("string"), F.lit(seed + 999))
    position = F.percent_rank().over(Window.partitionBy(LABEL_COL).orderBy(order))
    # percent_rank reaches exactly 1.0 for the last row of each group, which would
    # give floor(1.0 * 3) = 3 - a fold that does not exist. `least` caps it at 2,
    # so every single row lands in a real fold.
    fold_number = F.least(F.floor(position * n_folds).cast("int"), F.lit(n_folds - 1))
    return df.withColumn("stack_fold", fold_number)


def oof_churn_probability(classifier, prob_col):
    """Out-of-fold churn probabilities for one base model inside the training split.

    For each of the 3 folds: fit preparation + model on the OTHER two folds (so the
    preparation is also fitted fold by fold), then predict the held-out fold.
    Every training row ends up with a probability from a model that never saw it.
    """
    folded = add_stack_fold(train).cache()
    parts = []
    for fold in range(N_STACK_FOLDS):
        taught = folded.filter(F.col("stack_fold") != fold)
        held_out = folded.filter(F.col("stack_fold") == fold)
        fold_model = make_pipeline(classifier).fit(taught)
        parts.append(
            fold_model.transform(held_out)
            .select(KEY_COL, LABEL_COL,
                    vector_to_array("probability")[1].cast("double").alias(prob_col))
        )
    out = parts[0]
    for part in parts[1:]:
        out = out.unionByName(part)
    return out


# The two base models from Step 7 with their tuned settings, built fresh
# (they are fitted once per fold, so we pass the settings, not fitted models).
stack_gbt = GBTClassifier(labelCol=LABEL_COL, seed=SEED, **GBT_BEST)
stack_rf = RandomForestClassifier(labelCol=LABEL_COL, seed=SEED, **RF_BEST)

oof_gbt = oof_churn_probability(stack_gbt, "p_gbt")
oof_rf = oof_churn_probability(stack_rf, "p_rf")

meta_train = (
    oof_gbt.join(oof_rf, [KEY_COL, LABEL_COL], "inner")
    .select(KEY_COL, LABEL_COL, "p_gbt", "p_rf")
)
print(f"Out-of-fold training rows for the meta-model: {meta_train.count():,}")
meta_train = VectorAssembler(
    inputCols=["p_gbt", "p_rf"], outputCol="features"
).transform(meta_train).cache()

In [0]:
# 1) Meta-model: a plain logistic regression on the two out-of-fold probabilities.
meta_model = LogisticRegression(labelCol=LABEL_COL).fit(meta_train)
print(
    "Meta-model: p_churn = sigmoid("
    f"{meta_model.intercept:.3f} + {meta_model.coefficients[0]:.3f}*p_gbt"
    f" + {meta_model.coefficients[1]:.3f}*p_rf)"
)

# 2) Refit both base models on the FULL training split; their probabilities on any
#    new data are the meta-model's inputs.
base_gbt_full = make_pipeline(
    GBTClassifier(labelCol=LABEL_COL, seed=SEED, **GBT_BEST)
).fit(train)
base_rf_full = make_pipeline(
    RandomForestClassifier(labelCol=LABEL_COL, seed=SEED, **RF_BEST)
).fit(train)


def meta_features(data):
    """Churn probabilities of both base models for any customer set -> one vector."""
    gbt_p = (
        base_gbt_full.transform(data)
        .select(KEY_COL, LABEL_COL, vector_to_array("probability")[1].cast("double").alias("p_gbt"))
    )
    rf_p = (
        base_rf_full.transform(data)
        .select(KEY_COL, vector_to_array("probability")[1].cast("double").alias("p_rf"))
    )
    joined = gbt_p.join(rf_p, KEY_COL).select(KEY_COL, LABEL_COL, "p_gbt", "p_rf")
    return VectorAssembler(inputCols=["p_gbt", "p_rf"], outputCol="features").transform(joined)


# 3) Score the ensemble. "Training" here means: the meta-model applied to the
#    out-of-fold features of the training rows (see the Result cell for the caveat).
# Re-running this cell (e.g. while experimenting) replaces its row instead of adding a duplicate.
results[:] = [r for r in results if r["model"] != "Stacking (GBT + RF)"]
stack_train_preds = meta_model.transform(meta_train)
stack_val_preds = meta_model.transform(meta_features(val))
score_row("Stacking (GBT + RF)", stack_train_preds, stack_val_preds, "tuned")

stack_compare = pd.DataFrame([
    r for r in results
    if r["stage"] == "tuned"
    and r["model"] in ("Random Forest", "Gradient-Boosted Trees", "Stacking (GBT + RF)")
])
display(spark.createDataFrame(stack_compare[[
    "model", "train_auc_pr", "val_auc_pr", "train_val_gap",
    "val_precision_churn", "val_recall_churn", "val_f1_churn",
]]))
show_confusion(stack_val_preds, "Stacking (GBT + RF) on validation")

**Result — stacking wins on validation:**

| Model | Training AUC-PR | Validation AUC-PR | Gap |
|---|---|---|---|
| **Stacking (GBT + RF)** | 0.697 | **0.658** | **0.039** |
| Gradient-Boosted Trees (tuned) | 0.733 | 0.654 | 0.079 |
| Random Forest (tuned) | 0.885 | 0.651 | 0.234 |

- The meta-model learned `p_churn = sigmoid(−3.089 + 3.641·p_gbt + 2.952·p_rf)`: **both probabilities get positive, similar weights** — the two models really do make different mistakes, and the combination exploits it. With only two inputs, the meta-model itself has nothing to overfit.
- **Validation AUC-PR 0.658** — the best score in the whole notebook — with the **smallest gap (0.039)** of the strong models.
- A caution on the “training” column: 0.697 is the meta-model applied to *out-of-fold* probabilities, so it reflects honest out-of-fold base-model performance, not memorisation — which is why the gap is so small. The validation score remains the only decision basis.
- At threshold 0.5 on validation: precision 75%, **recall 46% (best of all models)**, F1 0.569 — it catches 140 of the 305 leavers.